In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap
import joblib
import os

print(f'Rebuilding data pipeline...')
MODELS_DIR = '../outputs/models'
OUTPUT_DIR = '../outputs/figures'
 
BG    = '#ffffff'
MUTED = '#888888'
TEXT  = '#222222'
WELL_COLOURS = {
    '15/9-F-1 C':  '#f39c12',
    '15/9-F-11':   '#3498db',
    '15/9-F-12':   '#2ecc71',
    '15/9-F-14':   '#e74c3c',
    '15/9-F-15 D': '#9b59b6',
    '15/9-F-5':    '#1abc9c',
}
AGG_RULES = {
    'OIL_RATE_NORM':         'mean',
    'AVG_WHP_P':             'mean',
    'AVG_DOWNHOLE_PRESSURE': 'mean',
    'WCT':                   'mean',
    'GOR':                   'median',
    'DRAWDOWN':              'mean',
    'DAYS_ON_PROD':          'max',
    'ON_STREAM_HRS':         'sum',
}

df = pd.read_parquet('../data/processed/cleaned.parquet')
df['DATEPRD'] = pd.to_datetime(df['DATEPRD'])
df = df.set_index('DATEPRD').sort_index()
df['OIL_RATE_NORM'] = df.groupby('NPD_WELL_BORE_NAME')['OIL_RATE_NORM'].transform(
    lambda x : x.clip(upper= x.quantile(0.99))
)

TARGET_WELLS = ['15/9-F-12', '15/9-F-14']
monthly_well_data = {}
for well_name,well_df in df.groupby('NPD_WELL_BORE_NAME'):
    if(well_name not in TARGET_WELLS):
        print(f'this well has no clean data')
        continue
    cols_availble = {k: v for k, v in AGG_RULES.items() if k in well_df.columns}
    monthly = well_df[list(cols_availble.keys())].resample('ME').agg(
        cols_availble
    ).dropna(subset=['OIL_RATE_NORM'])
    monthly_well_data[well_name] = monthly

def engineer_features(month_df):
    df_feature = month_df.copy()
    df_feature['lag_1_oil'] = df_feature['OIL_RATE_NORM'].shift(1)
    df_feature['lag_3_oil'] = df_feature['OIL_RATE_NORM'].shift(3)
    df_feature['lag_6_oil'] = df_feature['OIL_RATE_NORM'].shift(6)
    df_feature['lag_1_wct'] = df_feature['WCT'].shift(1) if 'WCT' in df_feature.columns else np.nan
    df_feature['lag_3_wct'] = df_feature['WCT'].shift(3) if 'WCT' in df_feature.columns else np.nan
    df_feature['lag_6_wct'] = df_feature['WCT'].shift(6) if 'WCT' in df_feature.columns else np.nan
    df_feature['lag_1_whp'] = df_feature['AVG_WHP_P'].shift(1) if 'AVG_WHP_P' in df_feature.columns else np.nan
    df_feature['rolling_3m'] = df_feature['OIL_RATE_NORM'].rolling(window=3, min_periods=2).mean()
    df_feature['rolling_6m'] = df_feature['OIL_RATE_NORM'].rolling(window=6, min_periods=3).mean()
    df_feature['month_over_month'] = df_feature['OIL_RATE_NORM'].pct_change().clip(-1, 1)
    df_feature['rate_vs_6m_avg'] = df_feature['lag_1_oil'] / (df_feature['rolling_6m'] + 1e-6)
    df_feature['wct_change'] = df_feature['WCT'].diff()
    df_feature['well_age'] = df_feature['DAYS_ON_PROD'] / 365
    return df_feature

FEATURE_COLS = [
    'lag_1_oil', 'lag_3_oil', 'lag_6_oil',
    'lag_1_wct', 'lag_3_wct', 'lag_6_wct',
    'lag_1_whp', 'rolling_3m', 'rolling_6m',
    'month_over_month', 'rate_vs_6m_avg',
    'WCT', 'AVG_WHP_P', 'DAYS_ON_PROD',
    'AVG_DOWNHOLE_PRESSURE', 'DRAWDOWN', 'GOR',
    'wct_change', 'well_age',
]

TARGET_COL = 'OIL_RATE_NORM'

well_splits = {}
for well_name in TARGET_WELLS:
    if well_name not in monthly_well_data:
        continue
    monthly  = monthly_well_data[well_name]
    df_feat  = engineer_features(monthly).dropna(subset=[TARGET_COL])
    peak_idx = df_feat[TARGET_COL].idxmax()
    peak_pos = df_feat.index.get_loc(peak_idx)
    decline  = df_feat.iloc[peak_pos:].copy()
    n_train  = int(len(decline) * 0.8)
 
    feat_available = [c for c in FEATURE_COLS if c in decline.columns]
    train_clean = decline.iloc[:n_train][feat_available + [TARGET_COL]].dropna()
    test_clean  = decline.iloc[n_train:][feat_available + [TARGET_COL]].dropna()
 
    well_splits[well_name] = {
        'decline':         decline,
        'train_clean':     train_clean,
        'test_clean':      test_clean,
        'feat_available':  feat_available,
        'n_train':         n_train,
        'X_train': train_clean[feat_available],
        'y_train': train_clean[TARGET_COL],
        'X_test':  test_clean[feat_available],
        'y_test':  test_clean[TARGET_COL],
    }
 
print(f"Data rebuilt for {list(well_splits.keys())}")
 
 
print("\nLoading saved XGBoost models...")
models = {}
for well_name in TARGET_WELLS:
    model_path = f'{MODELS_DIR}/xgb_{well_name.replace("/", "-").replace(" ", "_")}.joblib'
    if os.path.exists(model_path):
        models[well_name] = joblib.load(model_path)
        print(f"  Loaded: {model_path}")
    else:
        print(f"  NOT FOUND: {model_path} — retrain from 03_xgboost.py first")

print("\nComputing SHAP values...")
shap_data = {}
 
for well_name in TARGET_WELLS:
    if well_name not in models or well_name not in well_splits:
        print(f"  Skipping {well_name} — model or data missing")
        continue
 
    model  = models[well_name]
    splits = well_splits[well_name]
 
    print(f"  Computing SHAP for {well_name}...")
 
    explainer = shap.TreeExplainer(model)
 
    shap_train = explainer(splits['X_train'])
 
    shap_test  = explainer(splits['X_test'])
 
    shap_data[well_name] = {
        'explainer':   explainer,
        'shap_train':  shap_train,   
        'shap_test':   shap_test,
        'X_train':     splits['X_train'],
        'X_test':      splits['X_test'],
        'y_train':     splits['y_train'],
        'y_test':      splits['y_test'],
    }
 
    print(f"    SHAP matrix shape (train): {shap_train.values.shape}")
    print(f"    Baseline (expected value): {explainer.expected_value:.1f} Sm³/day")
    print(f"    This is the average prediction across all training rows.")
 
print("\nPlot 09: Beeswarm plots...")
 
fig, axes = plt.subplots(1, len(shap_data), figsize=(16, 8), facecolor=BG)
if len(shap_data) == 1:
    axes = [axes]
 
for ax, (well_name, sd) in zip(axes, shap_data.items()):
    plt.sca(ax)
 
    shap.plots.beeswarm(
        sd['shap_train'],
        max_display=12,
        show=False,
        color_bar=True,
    )
 
    ax.set_title(f'SHAP Beeswarm — {well_name}\nGlobal feature impact on all training predictions',
                 fontsize=10, fontweight='bold', color=TEXT, pad=8)
    ax.tick_params(colors=MUTED, labelsize=8)
 
plt.tight_layout()
path = f'{OUTPUT_DIR}/09_shap_beeswarm.png'
fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print(f"  Saved → {path}")
 


 
print("\nPlot 10: Waterfall plots (local explanation of best and worst predictions)...")
 
for well_name, sd in shap_data.items():
    colour  = WELL_COLOURS.get(well_name, '#333333')
    y_test  = sd['y_test']
    X_test  = sd['X_test']
    model   = models[well_name]
 
    y_pred  = model.predict(X_test)
    errors  = np.abs(y_pred - y_test.values)
 
    worst_idx = np.argmax(errors)
    best_idx  = np.argmin(errors)
 
    fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor=BG)
    fig.suptitle(f'SHAP Waterfall — {well_name}\nHow XGBoost built each prediction from baseline',
                 fontsize=13, fontweight='bold', color=TEXT, y=1.01)
 
    for ax, idx, label in [
        (axes[0], worst_idx, f'WORST prediction\n(error={errors[worst_idx]:.0f} Sm³/day)'),
        (axes[1], best_idx,  f'BEST prediction\n(error={errors[best_idx]:.0f} Sm³/day)'),
    ]:
        plt.sca(ax)
 
        shap.plots.waterfall(
            sd['shap_test'][idx],
            max_display=10,
            show=False,
        )
 
        # Add actual vs predicted annotation
        actual_val = y_test.values[idx]
        pred_val   = y_pred[idx]
        date_str   = y_test.index[idx].strftime('%b %Y')
 
        ax.set_title(f'{label}\nDate: {date_str} | Actual: {actual_val:.0f} | Predicted: {pred_val:.0f}',
                     fontsize=9, fontweight='bold', color=TEXT, pad=6)
        ax.tick_params(colors=MUTED, labelsize=8)
 
    plt.tight_layout()
    safe_name = well_name.replace('/', '-').replace(' ', '_')
    path = f'{OUTPUT_DIR}/10_shap_waterfall_{safe_name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close()
    print(f"  Saved → {path}")
 